# Chapitre 19 · La frontière

Notebook du chapitre 19 de *Construire un LLM de zéro*.

**Comment travailler.** La leçon est complète et exécutable de bout en bout : lis,
exécute, triture. Ce chapitre est **conceptuel** : on ne reconstruit plus un LLM entier,
on touche chaque idée de la frontière avec une **démo numérique de quelques lignes**
(la fiche technique, RoPE, RMSNorm, GQA, SwiGLU, MoE, la mémoire de l'attention) et
un **cas qui échoue** (l'extrapolation de longueur). À la fin, la section
**Exercices** : cinq défis à trous, du plus simple au plus costaud, validés par des `assert`.

Tout tourne hors ligne, sans GPU, en moins d'une minute.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("torch", torch.__version__)

## 1. Le fil rouge : la fiche technique de Qwen3-30B-A3B

Tout modèle publié sur Hugging Face embarque un fichier `config.json` : sa **fiche
technique**. Voici, simplifiée, celle de Qwen3-30B-A3B. Presque chaque champ pointe
vers une section de ce chapitre : chaque champ nouveau est la trace d'une innovation.

In [ ]:
config_qwen3_30b_a3b = {
    "hidden_size": 2048,             # d_model, la largeur des vecteurs
    "num_hidden_layers": 48,         # 48 blocs empiles (ton GPT : 2)
    "num_attention_heads": 32,       # 32 tetes de requete
    "num_key_value_heads": 4,        # 4 tetes K/V seulement  -> section 4 (GQA)
    "head_dim": 128,                 # dimension d'une tete
    "hidden_act": "silu",            # activation de la FFN     -> section 5 (SwiGLU)
    "num_experts": 128,              # 128 experts stockes      -> section 6 (MoE)
    "num_experts_per_tok": 8,        # 8 experts actifs / token -> section 6 (MoE)
    "moe_intermediate_size": 768,    # d_ff de chaque expert
    "rope_theta": 1000000,           # base RoPE                -> section 2 (RoPE)
    "rms_norm_eps": 1e-6,            # epsilon de la norme      -> section 3 (RMSNorm)
    "max_position_embeddings": 40960,  # fenetre de positions   -> section 8 (contexte long)
    "vocab_size": 151936,            # 151 936 tokens (ton GPT : 81)
}

for champ, valeur in config_qwen3_30b_a3b.items():
    print(f"{champ:>25} : {valeur}")

In [ ]:
# premiere lecture de mecanicien : trois deductions directes depuis la fiche
c = config_qwen3_30b_a3b
print(f"GQA : {c['num_attention_heads']} tetes Q pour {c['num_key_value_heads']} tetes K/V"
      f" -> cache K/V divise par {c['num_attention_heads'] // c['num_key_value_heads']}")
print(f"MoE : {c['num_experts_per_tok']} experts actifs sur {c['num_experts']} stockes"
      f" -> {100 * c['num_experts_per_tok'] / c['num_experts']:.1f}% du FFN calcule par token")
print(f"vocabulaire : {c['vocab_size']:,} tokens (ton GPT du chapitre 10 : 81)".replace(",", " "))

## 2. RoPE : encoder la position par une rotation

RoPE (*Rotary Position Embedding*, encodage de position par rotation) découpe un
vecteur en paires de coordonnées et **fait tourner** chaque paire d'un angle
proportionnel à la position. On prend une seule paire `(x, y)`, une petite flèche 2D,
et on la fait tourner d'un angle `m * theta` à la position `m`.

In [ ]:
def rot(v, theta):
    # rotation 2D d'un angle theta (sans changer la longueur)
    c, s = math.cos(theta), math.sin(theta)
    x, y = v
    return (x * c - y * s, x * s + y * c)

v = (1.0, 0.0)      # la fleche de depart
theta = 0.5         # la vitesse de rotation de cette paire
for m in [0, 1, 2, 3]:
    rx, ry = rot(v, m * theta)
    print(f"position m={m}: ({rx:+.3f}, {ry:+.3f})   longueur = {math.hypot(rx, ry):.3f}")

La longueur ne bouge **jamais** (toujours 1.000) : une rotation ne fait que changer
l'angle. C'est toute l'astuce de RoPE.

Maintenant la propriété qui rend RoPE si utile : le **produit scalaire** entre une
requête tournée à la position `m` et une clé tournée à la position `n` ne dépend que
de la **distance** `m - n`, pas des positions absolues.

In [ ]:
q = torch.randn(2)
k = torch.randn(2)

def rot_t(v, ang):
    c, s = math.cos(ang), math.sin(ang)
    return torch.tensor([v[0] * c - v[1] * s, v[0] * s + v[1] * c])

def score(m, n):
    return torch.dot(rot_t(q, m * theta), rot_t(k, n * theta)).item()

print(f"score(m=2,  n=5)  = {score(2, 5):+.4f}    (distance m-n = -3)")
print(f"score(m=12, n=15) = {score(12, 15):+.4f}    (meme distance m-n = -3)")
print(f"score(m=2,  n=8)  = {score(2, 8):+.4f}    (distance differente : -6)")

Les deux premiers scores sont **identiques** : mêmes vecteurs de base, même distance
`m - n = -3`, donc même score, que ce soit aux positions `(2, 5)` ou `(12, 15)`. Le
troisième, à une distance différente, donne un autre score. C'est la propriété de
**position relative** de RoPE : le modèle apprend des relations de distance, ce qui
l'aide à extrapoler à des contextes plus longs.

## 3. RMSNorm vs LayerNorm sur un vecteur

LayerNorm **centre** (retire la moyenne) puis **réduit** (divise par l'écart-type),
avec deux paramètres appris (`gamma` d'échelle, `beta` de biais). RMSNorm observe que
le centrage n'apporte presque rien : on peut se contenter de **diviser par la
magnitude** du vecteur (sa RMS, *root mean square*), et de garder le seul `gamma`.

In [ ]:
x = torch.tensor([2.0, -1.0, 0.5, 3.0, -2.5, 1.0, 0.0, -0.5])

# LayerNorm : centrer puis reduire
mu = x.mean()
sigma = x.std(unbiased=False)
layer_norm = (x - mu) / torch.sqrt(sigma ** 2 + 1e-6)

# RMSNorm : juste diviser par la RMS (pas de centrage, pas de biais)
rms = torch.sqrt((x ** 2).mean() + 1e-6)
rms_norm = x / rms

print(f"x         = {[round(v, 2) for v in x.tolist()]}")
print(f"moyenne   = {mu.item():.4f}   ecart-type = {sigma.item():.4f}   RMS = {rms.item():.4f}")
print(f"LayerNorm = {[round(v, 3) for v in layer_norm.tolist()]}")
print(f"RMSNorm   = {[round(v, 3) for v in rms_norm.tolist()]}")

Les deux sorties se ressemblent : RMSNorm produit un résultat du même ordre que
LayerNorm, en sautant le centrage et le biais. Le gain, c'est la simplicité.
Comptons les paramètres.

In [ ]:
d = 4096
params_layernorm = 2 * d    # gamma ET beta
params_rmsnorm = d          # gamma seul
print(f"pour d = {d} :")
print(f"  LayerNorm : {params_layernorm} parametres (gamma + beta)")
print(f"  RMSNorm   : {params_rmsnorm} parametres (gamma)  ->  moitie moins")

## 4. GQA : les shapes des têtes partagées

GQA (*Grouped-Query Attention*, attention par groupes de requêtes) garde beaucoup de
têtes pour la requête `Q`, mais **peu** de têtes pour les clés `K` et valeurs `V`.
Chaque tête K/V est ensuite **partagée** par plusieurs têtes Q. Le réflexe `shape`
raconte toute l'histoire.

In [ ]:
d_model, n_q, n_kv, seq = 512, 8, 2, 6
d_head = d_model // n_q

x = torch.randn(1, seq, d_model)
W_q = nn.Linear(d_model, n_q * d_head, bias=False)
W_k = nn.Linear(d_model, n_kv * d_head, bias=False)

Q = W_q(x).view(1, seq, n_q, d_head).transpose(1, 2)    # (1, n_q, seq, d_head)
K = W_k(x).view(1, seq, n_kv, d_head).transpose(1, 2)   # (1, n_kv, seq, d_head)

print(f"Q shape = {tuple(Q.shape)}   ({n_q} tetes de requete)")
print(f"K shape = {tuple(K.shape)}   ({n_kv} tetes cle/valeur seulement)")

# avant l'attention, on repete les tetes K/V pour matcher les tetes Q
K_rep = K.repeat_interleave(n_q // n_kv, dim=1)
print(f"K repete = {tuple(K_rep.shape)}   (1 tete K/V partagee par {n_q // n_kv} tetes Q)")
print(f"-> le cache K/V est {n_q // n_kv}x plus petit qu'en attention multi-tetes classique")

On stocke **2 têtes** K/V au lieu de 8 : le cache K/V est divisé par 4, pour une
qualité quasi inchangée. C'est le compromis que presque tous les gros modèles ont adopté.

## 5. SwiGLU : une porte dans le FFN

Le FFN moderne remplace l'activation fixe par une **porte** que le modèle dose
entrée par entrée : la branche porte passe par **Swish** (le `silu` de la fiche),
`z * sigmoide(z)`. Et la règle des 8/3 garde le FFN au même prix malgré la
troisième matrice.

In [ ]:
# Swish (alias SiLU) : z * sigmoide(z), la courbe qui passe dans la porte
z = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
swish = z * torch.sigmoid(z)
for zi, si in zip(z.tolist(), swish.tolist()):
    print(f"Swish({zi:+.1f}) = {si:+.3f}")

# la regle des 8/3 : trois matrices plus etroites au prix de deux larges
d_mod = 4096
d_ff_gelu = 4 * d_mod           # FFN classique : 2 matrices, d_ff = 4 * d_model
d_ff_swiglu = 8 * d_mod // 3    # SwiGLU : 3 matrices, d_ff ~ 8/3 * d_model
p_gelu = 2 * d_mod * d_ff_gelu
p_swiglu = 3 * d_mod * d_ff_swiglu
print(f"\nFFN GELU   : 2 matrices de largeur {d_ff_gelu}  -> {p_gelu:,} parametres".replace(",", " "))
print(f"FFN SwiGLU : 3 matrices de largeur {d_ff_swiglu}  -> {p_swiglu:,} parametres (quasi identique)".replace(",", " "))

## 6. MoE : un routage jouet sur 4 experts

Le MoE (*Mixture of Experts*, mélange d'experts) remplace le gros FFN unique par
plusieurs FFN plus petits (les **experts**) et un **routeur** qui, pour chaque token,
choisit les `k` meilleurs. On simule un routeur qui sort 4 scores bruts.

In [ ]:
N, k = 4, 2
logits = torch.tensor([2.0, 0.0, 1.0, 0.0])   # le routeur score les 4 experts

probs = F.softmax(logits, dim=-1)             # scores -> probabilites
top_w, top_i = probs.topk(k)                  # on garde les k meilleurs
top_w_norm = top_w / top_w.sum()              # on renormalise leurs poids

print(f"logits routeur    = {logits.tolist()}")
print(f"probas softmax    = {[round(v, 3) for v in probs.tolist()]}")
print(f"top-{k} experts    = {top_i.tolist()}   poids = {[round(v, 3) for v in top_w_norm.tolist()]}")
print(f"experts calcules  : {k}/{N} = {100 * k // N}% du FFN (activation sparse)")

Seuls **2 experts sur 4** sont calculés pour ce token. Les deux autres ne travaillent
jamais pour lui : c'est l'**activation sparse** (creuse). On stocke beaucoup de
paramètres, on n'en active qu'une fraction. C'est ce qui permet à un modèle comme
Mixtral 8x7B d'avoir environ 47 milliards de paramètres au total mais de n'en calculer
que 13 milliards par token.

## 7. Flash attention : le mur mémoire, chiffré

L'attention compare chaque token à tous les autres : pour `n` tokens, une matrice de
scores `n x n`, un coût **quadratique**. Les approches « linéaires » résument le passé
dans un état de taille fixe, un coût **linéaire** en `n`. Chiffrons l'écart.

In [ ]:
d = 128   # dimension d'une tete
print(f"{'n (tokens)':>12} | {'scores n*n':>18} | {'lineaire n*d':>15} | {'ratio':>8}")
print("-" * 62)
for n in [1024, 8192, 128000]:
    quad = n * n
    lin = n * d
    print(f"{n:>12,} | {quad:>18,} | {lin:>15,} | {quad // lin:>6,}x".replace(",", " "))

À 128 000 tokens, la matrice quadratique pèse **mille fois** plus que l'état linéaire.
C'est ce mur qui motive Flash Attention (calculer l'attention exacte par blocs, sans
jamais former la matrice `n x n` en entier dans la grande mémoire du GPU) et les
attentions linéaires ou hybrides des modèles récents.

## 8. Le cas qui échoue : extrapoler sans RoPE scaling

Voici le « cas qui échoue », exécuté sous tes yeux. On entraîne un petit GPT avec RoPE
sur des séquences **courtes**, puis on lui demande de lire des séquences **six fois
plus longues** que tout ce qu'il a vu. Sans adaptation de RoPE, les positions inédites
produisent des rotations incohérentes et la perplexité **explose**. Puis on « étire »
la base de RoPE et on regarde la perplexité redescendre en partie.

On garde un fil rouge francophone : le zémidjan de Sètondji et les mangues d'Awa.

In [ ]:
# petit corpus tres regulier : le contenu des sequences longues reste "connu",
# on isole ainsi l'effet de la POSITION.
motif = "le zemidjan de setondji roule vite. awa vend des mangues au marche. "
corpus = motif * 200

chars = sorted(set(corpus))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])
print(f"corpus : {len(corpus)} caracteres, vocabulaire de {V} tokens")

In [ ]:
# --- RoPE : precalcul des cos/sin et application (convention rotate_half) ---
def rope_freqs(dim, L, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    pos = torch.arange(L).float()
    fr = torch.outer(pos, inv)
    emb = torch.cat([fr, fr], dim=-1)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
# --- un GPT minuscule, une seule couche, RoPE dans l'attention ---
TRAIN_LEN = 24
d_model, n_head = 64, 4
d_head = d_model // n_head

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, d_model)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                nn.Linear(4 * d_model, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V, bias=False)

    def forward(self, idx, cos, sin):
        B, T = idx.shape
        h = self.emb(idx)
        x = self.ln1(h)
        q = self.Wq(x).view(B, T, n_head, d_head).transpose(1, 2)
        k = self.Wk(x).view(B, T, n_head, d_head).transpose(1, 2)
        v = self.Wv(x).view(B, T, n_head, d_head).transpose(1, 2)
        c = cos[:T].unsqueeze(0).unsqueeze(0)
        s = sin[:T].unsqueeze(0).unsqueeze(0)
        q = apply_rope(q, c, s)
        k = apply_rope(k, c, s)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        att = att.masked_fill(mask, float('-inf')).softmax(-1)
        o = (att @ v).transpose(1, 2).contiguous().view(B, T, d_model)
        h = h + self.Wo(o)
        h = h + self.ff(self.ln2(h))
        return self.head(h)

model = TinyGPT()
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
cos_tr, sin_tr = rope_freqs(d_head, TRAIN_LEN)

for step in range(600):
    ix = torch.randint(0, len(data) - TRAIN_LEN - 1, (32,))
    xb = torch.stack([data[i:i + TRAIN_LEN] for i in ix])
    yb = torch.stack([data[i + 1:i + TRAIN_LEN + 1] for i in ix])
    logits = model(xb, cos_tr, sin_tr)
    loss = F.cross_entropy(logits.reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()

print(f"loss finale a l'entrainement (T = {TRAIN_LEN}) : {loss.item():.3f}")

In [ ]:
@torch.no_grad()
def perplexite(T, scale=1.0):
    model.eval()
    base = 10000.0 * (scale ** (d_head / (d_head - 2)))   # base RoPE etiree si scale > 1
    cos, sin = rope_freqs(d_head, T, base=base)
    total_nll, ntok = 0.0, 0
    n_blocs = min((len(data) - 1) // T, 20)
    for i in range(n_blocs):
        xb = data[i * T:(i + 1) * T].unsqueeze(0)
        yb = data[i * T + 1:(i + 1) * T + 1].unsqueeze(0)
        lo = model(xb, cos, sin)
        total_nll += F.cross_entropy(lo.reshape(-1, V), yb.reshape(-1), reduction='sum').item()
        ntok += yb.numel()
    return math.exp(total_nll / ntok)

ppl_vue    = perplexite(TRAIN_LEN, scale=1.0)
ppl_extra  = perplexite(TRAIN_LEN * 6, scale=1.0)   # 6x plus long, RoPE naif
ppl_scaled = perplexite(TRAIN_LEN * 6, scale=6.0)   # base RoPE etiree

print(f"perplexite a T = {TRAIN_LEN} (longueur vue)              : {ppl_vue:.2f}")
print(f"perplexite a T = {TRAIN_LEN * 6} SANS scaling (extrapolation) : {ppl_extra:.2f}")
print(f"perplexite a T = {TRAIN_LEN * 6} AVEC base RoPE etiree        : {ppl_scaled:.2f}")
print(f"(perplexite d'un modele au hasard : {V})")

Lis les trois chiffres. À la longueur vue à l'entraînement, la perplexité est basse :
le modèle prédit presque parfaitement. Dès qu'on dépasse cette longueur **sans rien
changer**, elle fait plus que doubler : le modèle perd le fil, parce que RoPE tourne à
des angles qu'il n'a jamais rencontrés. En **étirant la base** de RoPE (l'idée derrière
le NTK-aware scaling et YaRN), on ralentit les rotations pour que les positions
lointaines restent distinguables, et la perplexité **redescend** en partie.

C'est exactement le problème que résolvent les contextes longs de 128k tokens et plus :
on ne réentraîne pas tout, on adapte RoPE.

## Exercices

À toi de jouer : cinq exercices, du plus simple (●) au plus costaud (●●●), un par
démo de la leçon. Chaque cellule marquée `# TODO(toi)` contient un trou ; complète-le,
puis exécute la cellule de validation (`assert`) qui suit : si elle passe sans erreur,
c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · RMSNorm à la main — niveau ●

LayerNorm centre puis réduit. RMSNorm saute le centrage : elle divise simplement
par la RMS du vecteur. Calcule-la sur ce nouveau vecteur, sans regarder la section 3.

In [ ]:
x_ex = torch.tensor([1.0, -2.0, 3.0, 0.5, -1.5, 2.5, 0.0, -3.0])

# TODO(toi) : calcule la RMS = racine( moyenne( x_ex^2 ) + 1e-6 )
#             puis rms_norm_ex = x_ex / rms_ex   (pas de centrage, pas de biais)
rms_ex = ...
rms_norm_ex = ...

print(f"RMS = {rms_ex.item():.4f}")
print(f"RMSNorm = {[round(v, 3) for v in rms_norm_ex.tolist()]}")

In [ ]:
# Validation : RMSNorm.
assert abs(rms_ex.item() - 1.9922) < 1e-3, "la RMS devrait valoir environ 1.9922"
assert abs(rms_norm_ex[2].item() - 1.506) < 1e-2, "verifie rms_norm_ex = x_ex / rms_ex"
print("OK : RMSNorm calculee (et remarque : pas besoin de la moyenne)")

### Exercice 2 · La propriété de distance de RoPE — niveau ●

Vérifie la propriété clé de RoPE sur une nouvelle paire de vecteurs : le
**produit scalaire** entre une requête tournée à la position `m` et une clé
tournée à la position `n` ne doit dépendre que de la **distance** `m - n`.

In [ ]:
q_ex = torch.tensor([0.8, -0.6])
k_ex = torch.tensor([0.3, 0.9])
theta_ex = 0.5

def rot_ex(v, ang):
    c, s = math.cos(ang), math.sin(ang)
    return torch.tensor([v[0] * c - v[1] * s, v[0] * s + v[1] * c])

def score_ex(m, n):
    # TODO(toi) : tourne q_ex d'un angle m*theta_ex, k_ex d'un angle n*theta_ex,
    #             puis retourne leur produit scalaire (torch.dot(...).item())
    return ...

s_proche = score_ex(3, 7)     # distance -4
s_loin   = score_ex(13, 17)   # meme distance -4, positions decalees de 10
s_autre  = score_ex(3, 10)    # distance differente : -7
print(f"score(3, 7)   = {s_proche:+.4f}")
print(f"score(13, 17) = {s_loin:+.4f}")
print(f"score(3, 10)  = {s_autre:+.4f}")

In [ ]:
# Validation : meme distance => meme score ; distance differente => score different
assert abs(s_proche - s_loin) < 1e-5, "les scores a distance egale devraient etre identiques"
assert abs(s_proche - s_autre) > 1e-3, "une distance differente devrait changer le score"
print("OK : le score d'attention ne depend que de la distance m - n")

### Exercice 3 · Répéter les têtes K/V (GQA) — niveau ●●

GQA stocke peu de têtes K/V et les **répète** au moment du calcul pour servir
les têtes Q. Ici, 12 têtes de requête et 3 têtes K/V : chaque tête K/V doit
servir un groupe de 4 têtes Q.

In [ ]:
n_q_ex, n_kv_ex = 12, 3
K_ex = torch.randn(1, n_kv_ex, 5, 16)   # (batch, tetes K/V, seq, d_head)

# TODO(toi) : repete les tetes K pour matcher les tetes Q.
#             Indice : K_ex.repeat_interleave(groupe, dim=?)  ou groupe = n_q_ex // n_kv_ex
#             Reflexe shape : la dimension des tetes est la dim 1.
K_rep_ex = ...

print(f"K_ex shape     = {tuple(K_ex.shape)}")
print(f"K_rep_ex shape = {tuple(K_rep_ex.shape)}")

In [ ]:
# Validation : les groupes partagent bien la meme tete K/V
assert K_rep_ex.shape == (1, n_q_ex, 5, 16), "K_rep_ex doit avoir autant de tetes que Q"
assert torch.equal(K_rep_ex[0, 0], K_rep_ex[0, 3]), "les 4 premieres tetes doivent partager la meme K"
assert torch.equal(K_rep_ex[0, 8], K_rep_ex[0, 11]), "les 4 dernieres tetes doivent partager la meme K"
assert not torch.equal(K_rep_ex[0, 3], K_rep_ex[0, 4]), "la tete 4 appartient au groupe suivant"
print(f"OK : 1 tete K/V partagee par {n_q_ex // n_kv_ex} tetes Q, cache divise par {n_q_ex // n_kv_ex}")

### Exercice 4 · Le routage MoE — niveau ●●

Le routeur d'une couche MoE sort un score par expert. Transforme en probabilités,
garde les `k` meilleurs, renormalise leurs poids. Nouveaux scores, même mécanique
que la section 6.

In [ ]:
N_ex, k_top = 4, 2
logits_ex = torch.tensor([0.5, 2.5, 1.5, -1.0])   # le routeur score les 4 experts

# TODO(toi) : 1) probs_ex = softmax des logits_ex
#             2) top_w_ex, top_i_ex = les k_top plus grosses probas et leurs indices (probs_ex.topk(k_top))
#             3) top_w_norm_ex = top_w_ex renormalise pour sommer a 1
probs_ex = ...
top_w_ex, top_i_ex = ...
top_w_norm_ex = ...

print(f"probas softmax  = {[round(v, 3) for v in probs_ex.tolist()]}")
print(f"top-{k_top} experts   = {top_i_ex.tolist()}   poids = {[round(v, 3) for v in top_w_norm_ex.tolist()]}")

In [ ]:
# Validation : routage top-2
assert sorted(top_i_ex.tolist()) == [1, 2], "les deux meilleurs experts sont 1 et 2"
assert abs(top_w_norm_ex.sum().item() - 1.0) < 1e-5, "les poids renormalises doivent sommer a 1"
assert abs(top_w_norm_ex[0].item() - 0.731) < 1e-2, "le poids du meilleur expert vaut environ 0.731"
print(f"OK : {k_top}/{N_ex} experts calcules pour ce token, les autres ne travaillent jamais")

### Exercice 5 · La perplexité au-delà de la longueur vue — niveau ●●●

Le petit GPT entraîné à la section 8 est encore en mémoire. Réécris toi-même la
mesure de perplexité : passe chaque bloc dans le modèle, accumule la NLL totale
et le nombre de tokens, et retrouve l'explosion puis la récupération partielle.

In [ ]:
@torch.no_grad()
def perplexite_ex(T, scale=1.0):
    model.eval()
    base = 10000.0 * (scale ** (d_head / (d_head - 2)))   # base RoPE etiree si scale > 1
    cos, sin = rope_freqs(d_head, T, base=base)
    total_nll, ntok = 0.0, 0
    n_blocs = min((len(data) - 1) // T, 20)
    for i in range(n_blocs):
        xb = data[i * T:(i + 1) * T].unsqueeze(0)
        yb = data[i * T + 1:(i + 1) * T + 1].unsqueeze(0)
        # TODO(toi) : lo = model(xb, cos, sin)
        #             accumule la NLL totale (cross_entropy reduction='sum') et le nombre de tokens
        total_nll += ...
        ntok += ...
    return math.exp(total_nll / ntok)

ppl_vue_ex    = perplexite_ex(TRAIN_LEN, scale=1.0)
ppl_extra_ex  = perplexite_ex(TRAIN_LEN * 6, scale=1.0)   # 6x plus long, RoPE naif
ppl_scaled_ex = perplexite_ex(TRAIN_LEN * 6, scale=6.0)   # base RoPE etiree

print(f"perplexite a T = {TRAIN_LEN} (longueur vue)              : {ppl_vue_ex:.2f}")
print(f"perplexite a T = {TRAIN_LEN * 6} SANS scaling (extrapolation) : {ppl_extra_ex:.2f}")
print(f"perplexite a T = {TRAIN_LEN * 6} AVEC base RoPE etiree        : {ppl_scaled_ex:.2f}")

In [ ]:
# Validation : le cas qui echoue, puis la recuperation partielle
assert ppl_vue_ex < 1.5, "a la longueur vue, le modele devrait predire presque parfaitement"
assert ppl_extra_ex > 2 * ppl_vue_ex, "au-dela de la longueur d'entrainement, la perplexite doit exploser"
assert ppl_scaled_ex < ppl_extra_ex, "la base RoPE etiree doit recuperer une partie de la degradation"
print("OK : extrapolation naive = perplexite qui explose ; RoPE scaling = recuperation partielle")

## Verdict

- **RoPE** : le score d'attention ne dépend que de la distance entre deux tokens.
- **RMSNorm** : diviser par la RMS suffit, pas besoin de centrer.
- **GQA** : 2 têtes K/V partagées par 8 têtes Q, cache divisé par 4.
- **SwiGLU** : une porte dosée par l'entrée, trois matrices au prix de deux (règle des 8/3).
- **MoE** : 2 experts sur 4 calculés par token, les autres dorment.
- **Mémoire** : quadratique contre linéaire, un rapport de mille à 128k tokens.
- **Le cas qui échoue** : sans RoPE scaling, dépasser la longueur d'entraînement ruine la perplexité.

Tu n'as rien reconstruit de A à Z ici : c'est le but d'un chapitre conceptuel. Les
corrigés des exercices sont dans
`solutions/partie_4_du_modele_a_lesprit/chapitre_19_la_frontiere_solution.ipynb`.
Direction le chapitre 20 : faire tourner un modèle pour de vrai.